In [ ]:
from dotenv import load_dotenv
from sqlalchemy import insert, select
from sqlalchemy.orm import Session
import polars as pl
import re
from datetime import datetime

# Importando a partir do pacote 'app'
from app.config import db_engine
from app.models import (
    Base,
    GeneralSearch
)
from app.services import OLXService
from app.utils import (
    read_search_locations_metadata,
    read_scraping_targets_metadata,
    read_additional_info_metadata
)

In [2]:
# Loads env variables from .env file
load_dotenv()

# Base list for general search
general_data_rows = []

# Creating tables
Base.metadata.create_all(db_engine)

# Date
today = datetime.now()

In [ ]:
async def scrap_olx(region_list: list[str]):
    olx_service = OLXService()
    await search_for_general_links(olx_service=olx_service, region_list=region_list)
    await olx_service.terminate_client()

async def search_for_general_links(olx_service: OLXService, region_list: list[str]):
    # Read the scraping targets metadata
    targets_metadata = read_scraping_targets_metadata().get("targets", [])
    itens = targets_metadata.items()

    for category, subcategories in itens:

        for subcategory, items in subcategories.items():
            append_subcategory = False
            if category in ["memory", "storage", "peripherals"]:
                append_subcategory = True
            if subcategory in ["motherboard"]:
                append_subcategory = True
            
            for item in items:
                item_name = item.get('pt', '')
                if append_subcategory or item_name == "":
                    item_name = subcategory.capitalize() + " " + item_name

                results = await olx_service.get_details_links(item_name)

                url = results.get('url', '')
                status = results.get('status', 400)
                links = results.get('links', [])

                region = re.search(r"https://([^.]+)", url).group(1)
                if status != 200:
                    print(f'Erro no processamento da url: {url}')
                else:
                    print(f'Sucesso no processamento da url: {url}')

                if region in region_list:
                    general_data_rows.append({
                        'category': category,
                        'subcategory': subcategory,
                        'item': item_name,
                        'url': url,
                        'status': status,
                        'links': links,
                        'datetime': today,
                    })

In [ ]:
# Search data
brazil_data_location = (
    read_search_locations_metadata()
    .get('search_locations', {})
    .get('brazil', {})
)

stores = (
    brazil_data_location
    .get('stores', [])
)

regions = (
    brazil_data_location
    .get('regions', [])
)

full_name_list = [region.get('full_name') for region in regions]
abreviation_list = [region.get('abreviation') for region in regions]

['sp', 'rj', 'rs', 'mg']


In [ ]:
await scrap_olx(abreviation_list)

INFO:httpx:HTTP Request: GET https://www.olx.com.br/brasil?q=Motherboard+B450 "HTTP/1.1 200 OK"


Sucesso no processamento da url: /brasil?q=Motherboard+B450


INFO:httpx:HTTP Request: GET https://www.olx.com.br/brasil?q=Motherboard+B550 "HTTP/1.1 200 OK"


Sucesso no processamento da url: /brasil?q=Motherboard+B550


INFO:httpx:HTTP Request: GET https://www.olx.com.br/brasil?q=Motherboard+H610 "HTTP/1.1 200 OK"


...

Sucesso no processamento da url: /brasil?q=Keyboard+Teclado+Sem+Fio


INFO:httpx:HTTP Request: GET https://www.olx.com.br/brasil?q=Monitor+LG+UltraGear+144Hz "HTTP/1.1 200 OK"


Sucesso no processamento da url: /brasil?q=Monitor+LG+UltraGear+144Hz


INFO:httpx:HTTP Request: GET https://www.olx.com.br/brasil?q=Monitor+AOC+Hero+144Hz "HTTP/1.1 200 OK"


Sucesso no processamento da url: /brasil?q=Monitor+AOC+Hero+144Hz


In [ ]:
# for store in stores:
#     store_name = store.get('name', '')
#     store_base_url = store.get('base_url', '')
    
#     print(f'Processing {store_name}')

    # if store_name == 'OLX':
    #     await scrap_olx(regions)

In [5]:
# Creating dataframe
general_df = pl.DataFrame(general_data_rows)

with Session(db_engine) as session:
    # Saving on sqlite
    session.execute(insert(GeneralSearch), general_df.to_dicts())
    session.commit()

In [26]:
with Session(db_engine) as session:
    # Trying to read
    stmt = select(GeneralSearch)
    search = session.scalars(stmt).first()
    print(search)

In [6]:
# from bs4 import BeautifulSoup
# from app.utils import get_httpx_client
# from app.scraper import scrape_url

# url = (df[0]['links'].item()[0])
# print(url)
# client = get_httpx_client(base_url="https://www.olx.com.br")

# try:
#     result = await scrape_url(url, client)
#     if result.get("status") == 200 and "body" in result:
#         soup = BeautifulSoup(result["body"], "html.parser")
#         print(result.get('body', {}))

# except Exception as e:
#     print(f'Error: {e}')